# Notebook 10 -- Shor's Algorithm

**Prerequisites:** Notebooks 01-08. Familiarity with QPE and the QFT.

Shor's algorithm is the most celebrated result in quantum computing: it factors
integers in polynomial time, breaking the RSA cryptosystem that underpins most
of today's internet security. Where the best known classical factoring algorithm
(the general number field sieve) runs in sub-exponential time
$O(\exp(n^{1/3} (\log n)^{2/3}))$, Shor's algorithm runs in $O(n^3)$ using
quantum phase estimation.

By the end of this notebook you will be able to:

1. **Describe** how Shor's algorithm reduces factoring to period finding.
2. **Implement** integer factoring using Goqu's shor package.
3. **Trace** the period-finding circuit structure and continued fraction extraction.

For quantum counting and amplitude estimation, see [Notebook 10b](10b-quantum-counting.ipynb).

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/shor"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## The Factoring Problem

Given a composite integer $N$, find two non-trivial factors $p$ and $q$ such
that $N = p \times q$. This is believed to be classically hard -- no
polynomial-time algorithm is known, and RSA encryption relies on this hardness.

For example, given $N = 15$, the factors are $3$ and $5$. For small numbers
this is trivial, but for numbers with hundreds of digits, classical algorithms
become infeasible.

## How Shor's Algorithm Works

Shor's algorithm converts factoring into **period finding** through the
following steps:

1. **Choose a random base** $a$ coprime to $N$.
2. **Find the period** $r$ of $f(x) = a^x \bmod N$ -- the smallest $r$ such
   that $a^r \equiv 1 \pmod{N}$.
3. **Extract factors**: if $r$ is even, compute
   $\gcd(a^{r/2} \pm 1, N)$ which, with high probability, yields a
   non-trivial factor.

The quantum speedup lives in step 2. Classically, finding the period requires
exponentially many evaluations of $f$. Quantum mechanically, we use **Quantum
Phase Estimation (QPE)** on the modular exponentiation unitary
$U|y\rangle = |ay \bmod N\rangle$.

The eigenvalues of $U$ are $e^{2\pi i s/r}$ for $s = 0, \ldots, r-1$, so QPE
measures a phase $\varphi \approx s/r$. A **continued fraction expansion** of
$\varphi$ recovers the period $r$, from which we compute the factors.

### Circuit Structure

The quantum circuit consists of:

1. A **phase register** of $t$ qubits initialized in superposition.
2. A **target register** of $\lceil\log_2 N\rceil$ qubits initialized to $|1\rangle$.
3. **Controlled modular exponentiation**: $\text{C-}U^{2^k}$ for each phase qubit $k$.
4. **Inverse QFT** on the phase register to extract the phase.
5. **Measurement** of the phase register, followed by classical post-processing.

### The period-finding circuit structure

Shor's algorithm uses QPE to find the period of modular exponentiation.
While the full circuit for large numbers is complex, we can demonstrate
the structure: a phase register in superposition, controlled modular
exponentiation, and inverse QFT.

For N=15 with base a=2, the modular exponentiation is U|x> = |2x mod 15>.
Let's trace through the key idea: QPE extracts the period from the phase.

In [2]:
%%
// Shor's algorithm structure: QPE on modular exponentiation
// For N=15, a=2: the period is r=4 (2^4 = 16 ≡ 1 mod 15)
//
// The full circuit has:
//   - Phase register (t qubits in superposition)
//   - Work register (holding the modular exponentiation result)
//   - Controlled-U^(2^k) operations
//   - Inverse QFT on the phase register
//
// We can verify the math: if r=4 and we use 3 phase bits,
// QPE should output phases that are multiples of 1/r = 1/4

fmt.Println("Shor's Algorithm for N=15")
fmt.Println("=========================")
fmt.Println()
fmt.Println("Step 1: Choose random base a=2")
fmt.Println("Step 2: Compute 2^k mod 15 for k=1,2,...,8:")
for k := 1; k <= 8; k++ {
	val := 1
	for i := 0; i < k; i++ {
		val = (val * 2) % 15
	}
	marker := ""
	if val == 1 {
		marker = " <-- period found! r=" + fmt.Sprint(k)
	}
	fmt.Printf("  2^%d mod 15 = %d%s\n", k, val, marker)
}
fmt.Println()
fmt.Println("Step 3: Period r=4. Factors from gcd(2^(r/2) ± 1, 15):")
fmt.Printf("  gcd(2^2 + 1, 15) = gcd(5, 15) = 5\n")
fmt.Printf("  gcd(2^2 - 1, 15) = gcd(3, 15) = 3\n")
fmt.Printf("  Therefore 15 = 3 × 5\n")

Shor's Algorithm for N=15

Step 1: Choose random base a=2
Step 2: Compute 2^k mod 15 for k=1,2,...,8:
  2^1 mod 15 = 2
  2^2 mod 15 = 4
  2^3 mod 15 = 8
  2^4 mod 15 = 1 <-- period found! r=4
  2^5 mod 15 = 2
  2^6 mod 15 = 4
  2^7 mod 15 = 8
  2^8 mod 15 = 1 <-- period found! r=8

Step 3: Period r=4. Factors from gcd(2^(r/2) ± 1, 15):
  gcd(2^2 + 1, 15) = gcd(5, 15) = 5
  gcd(2^2 - 1, 15) = gcd(3, 15) = 3
  Therefore 15 = 3 × 5


Now let's run the full algorithm using Goqu's `shor` package:

In [3]:
%%
ctx := context.Background()

// Factor N = 15 using Shor's algorithm
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("N = 15 = %d x %d\n", result.Factors[0], result.Factors[1])
fmt.Printf("Period: %d, Base: %d, Attempts: %d\n", result.Period, result.Base, result.Attempts)
fmt.Printf("\nVerification: %d x %d = %d\n",
	result.Factors[0], result.Factors[1],
	result.Factors[0]*result.Factors[1])

N = 15 = 3 x 5
Period: 0, Base: 9, Attempts: 2

Verification: 3 x 5 = 15


In [4]:
%%
ctx := context.Background()

// Run Shor's on N=15 again and display the order-finding circuit
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

if result.Circuit != nil {
	fmt.Println("Order-finding circuit (QPE + modular exponentiation):")
	gonbui.DisplayHTML(draw.SVG(result.Circuit))
} else {
	fmt.Println("Factor found classically (no circuit needed for this attempt).")
}

Factor found classically (no circuit needed for this attempt).


In [5]:
%%
ctx := context.Background()

// Run Shor's on N=15 and display the QPE measurement histogram.
// The circuit measures only the phase register; peaks correspond to
// phases s/r that encode the period r.
result, err := shor.Run(ctx, shor.Config{
	N:     15,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

if result.Circuit != nil {
	// Re-run the circuit to get measurement counts for visualization
	nTarget := int(math.Ceil(math.Log2(float64(15))))
	nPhase := 2 * nTarget
	nTotal := nPhase + nTarget
	sim := statevector.New(nTotal)
	counts, err := sim.Run(result.Circuit, 1000)
	if err != nil {
		panic(err)
	}
	fmt.Println("QPE measurement histogram for N=15:")
	fmt.Println("Peaks at multiples of 1/r reveal the period r.")
	svg := viz.Histogram(counts, viz.WithTitle("Shor's Algorithm: QPE Output (N=15)"))
	gonbui.DisplayHTML(svg)
} else {
	fmt.Println("Factor found classically -- no quantum circuit to visualize.")
}

QPE measurement histogram for N=15:
Peaks at multiples of 1/r reveal the period r.


Shor's Algorithm: QPE Output (N=15) 
 
 
 
 0 
 
 50 
 
 100 
 
 150 
 
 200 
 
 250 
 
 000100000000 
 
 000100000010 
 
 000100000011 
 
 001000000000 
 
 001000000001 
 
 001000000010 
 
 001000000011 
 
 111000000000 
 
 111000000001 
 
 111000000010 
 
 111000000011

## Predict, Then Verify

**Question:** What are the factors of $N = 21$?

**Prediction:** $21 = 3 \times 7$. Shor's algorithm should find these factors
by discovering the period of $a^x \bmod 21$ for some random base $a$.

For example, with $a = 2$:
- $2^1 \equiv 2$, $2^2 \equiv 4$, $2^3 \equiv 8$, $2^4 \equiv 16$,
  $2^5 \equiv 11$, $2^6 \equiv 1 \pmod{21}$
- Period $r = 6$, which is even.
- $\gcd(2^3 - 1, 21) = \gcd(7, 21) = 7$ and
  $\gcd(2^3 + 1, 21) = \gcd(9, 21) = 3$.

Let's verify with Goqu.

In [6]:
%%
ctx := context.Background()

// Factor N = 21
result, err := shor.Run(ctx, shor.Config{
	N:     21,
	Shots: 1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("N = 21 = %d x %d\n", result.Factors[0], result.Factors[1])
fmt.Printf("Period: %d, Base: %d, Attempts: %d\n", result.Period, result.Base, result.Attempts)
fmt.Printf("\nVerification: %d x %d = %d\n",
	result.Factors[0], result.Factors[1],
	result.Factors[0]*result.Factors[1])
fmt.Println("\nPrediction confirmed: 21 = 3 x 7.")

N = 21 = 3 x 7
Period: 0, Base: 18, Attempts: 1

Verification: 3 x 7 = 21

Prediction confirmed: 21 = 3 x 7.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does Shor's algorithm need a quantum computer for period finding but not for the GCD step?
2. What is the relationship between the period r and the factors of N?
3. Why does Shor's algorithm fail when r is odd, and how likely is this?

---

## Exercises

### Exercise 1 -- Factor N = 35

Run Shor's algorithm on $N = 35$. Verify that the returned factors are correct
and print the period and base that the algorithm discovered.

In [7]:
%%
// Exercise 1: Factor N = 35
// Expected factors: 5 and 7
//
// TODO: Run Shor's algorithm on N = 35
// result, err := shor.Run(ctx, shor.Config{
//     N:     ???,
//     Shots: 1000,
// })
//
// TODO: Print the factors, period, base, and number of attempts
// fmt.Printf("N = 35 = %d x %d\n", result.Factors[0], result.Factors[1])
//
// TODO: Verify correctness by checking result.Factors[0]*result.Factors[1] == 35
_ = context.Background()
fmt.Println("Uncomment the code above and fill in the config!")

Uncomment the code above and fill in the config!


## Key Takeaways

1. **Shor's algorithm** factors integers in polynomial time by reducing
   factoring to period finding. The quantum speedup comes from QPE on the
   modular exponentiation unitary, followed by continued fraction expansion
   to extract the period.

2. **The order-finding circuit** uses controlled modular exponentiation
   $\text{C-}U^{2^k}$ followed by the inverse QFT. The phase register
   encodes $s/r$ where $r$ is the period, and the factors are recovered via
   $\gcd(a^{r/2} \pm 1, N)$.

3. The algorithm is **probabilistic**: a random base $a$ might yield an odd
   period or trivial factors, but repeating with different bases succeeds
   with high probability after a few attempts.

4. While Shor's algorithm is polynomial in theory, the circuits for
   modular exponentiation of large numbers are very deep, making practical
   factoring of cryptographically relevant numbers a goal for future
   fault-tolerant quantum computers.

For quantum counting and amplitude estimation, see [Notebook 10b](10b-quantum-counting.ipynb).

---

**Next up:** Notebook 11, where we will explore quantum error correction
and how to protect quantum information from noise.